In [1]:
from transformers import pipeline

e:\GENAI H2S\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
summarizer = pipeline("text2text-generation", model="google/flan-t5-large")

Device set to use cuda:0


In [8]:
def get_final_verdict(claim, top_evidence):
    prompt = f"""
    You are a precise fact-checking AI. Your task is to analyze the trusted evidence and determine if it supports, refutes, or is insufficient to verify the user's claim.

    First, choose the single best label from the following options: [SUPPORTS, REFUTES, NOT ENOUGH INFO].
    Second, provide a concise, one-sentence explanation for your choice based on the evidence.

    Trusted Evidence: "{top_evidence}"

    User's Claim: "{claim}"

    Output your response in the following format:
    VERDICT: [Your chosen label]
    EXPLANATION: [Your one-sentence explanation]
    """
    result = summarizer(prompt, max_new_tokens=100)[0]['generated_text']
    return result_parser(result)

def result_parser(raw_text):
    try:
        lines = raw_text.strip().split('\n')
        verdict = lines[0].replace('VERDICT:', '').strip()
        explanation = lines[1].replace('EXPLANATION:', '').strip()
        return {
            'verdict': verdict,
            'explanation': explanation
        }
    except IndexError:
        return {
            'verdict': 'UNCLEAR',
            'explanation': raw_text.strip()
        }

In [9]:
user_claim = "The sky is green."
top_evidence_from_reranker = "Rayleigh scattering of sunlight in Earth's atmosphere causes diffuse sky radiation, which is why the sky appears blue."

verdict = get_final_verdict(user_claim, top_evidence_from_reranker)
print(verdict) 
# Expected Output: "REFUTES. The evidence states that the sky appears blue due to scientific principles."

{'verdict': 'UNCLEAR', 'explanation': "Rayleigh scattering of sunlight in Earth's atmosphere causes diffuse sky radiation, which is why the sky appears blue. The user's claim: The sky is green."}
